In [ ]:
import gymnasium as gym
import numpy as np
import pygad
import matplotlib.pyplot as plt
import mplcyberpunk
import time #do kontrolowania czasu 
import matplotlib


#1. Konfiguracja środowiska
# Plansza 4x4, wyłączam poślizg (is_slippery=False), aby ruch był w 100% przewidywalny dla algorytmu
env = gym.make('FrozenLake-v1',map_name="4x4", is_slippery=False, render_mode="rgb_array")

MAX_STEPS = 12 #Maksymalna liczba kroków na planszy 4x4 (najkrótsza droga to 6 kroków)

# Pusta macierz 4x4 do zliczenia odwiedzin  na kafelkach lodu
heatmap_data = np.zeros((4, 4))
# 2. Funkcja dopasowania (fitness function)
def fitness_func(ga_instance,solution, solution_idx):
    observation, info = env.reset(seed=42)
    total_reward = 0
    current_step = 0

    #przed startem ruchu wiemhy, ze agent stoi na polu startowym (indeks 0 -> wspołrzedne 0,0)
    #mapujemy indeks siatki (0-15) na współrzędne wiersz, kolumna (x, y)
    r, c = divmod(observation, 4) #obliczamy wiersz i kolumnę na podstawie indeksu
    heatmap_data[r, c] += 1 # zapisujemy krok na mapie ciepła

    # Przechodzenie przez sekwencje kroków zapisaną w genach osobnika
    for action in solution:
        observation, reward, terminated, truncated, info = env.step(int(action))
        total_reward += reward
        current_step += 1

        # zapisujemy kazda kolejna pozycje agenta na mapie ciepła
        r, c = divmod(observation, 4) 
        heatmap_data[r, c] += 1 #zliczamy odwiedziny na mapie ciepła

        if terminated:
            break

        # Jesli wpadliśmy do dziury (gra sie zakończyła, a nagroda to 0), przegrywamy
       # if terminated and reward == 0:
         #   break
        #Jesli doszlismy do celu (nagroda =1), przerywamy z sukcesem
        #if terminated and reward == 1:
          #  break
    

    #Obliczanie fitness 
    # Nagroda główna za dojscie do celu to 1.0
    # Jeśli osobnik nie doszedł, premiujemy go za to, ile kroków zdołał przeżyć bez wpadnięcia do dziury
    if total_reward == 1:
        # Sukces! Dajemy gigantyczny fitness, premiując dodatkowo mniejszą liczbę kroków (szybkość)
        fitness = 100.0 + (MAX_STEPS - current_step) # Im mniej kroków do celu, tym lepiej
    else:
        # Porażka (wpadł do dziury lub błądził). Fitness to liczba bezpiecznych kroków (max 12)
        fitness = current_step # Im więcej kroków przeżył, tym lepiej (max 12)

    return fitness

#3. Konfiguracja algorytmu genetycznego w PyGAS
ga_instance = pygad.GA(
    num_generations=100, # Liczba pokoleń
    num_parents_mating=10, # Liczba rodziców biorących udział w krzyżowaniu
    fitness_func=fitness_func, # Funkcja dopasowania
    sol_per_pop=40, # Liczba osobników w populacji
    num_genes=MAX_STEPS, # Liczba genów (kroków) w każdym osobniku -> chromosom ma 12 genów (ruchów)
    gene_space=[0, 1, 2, 3], # Możliwe ruchy: 0=lewo, 1=dół, 2=prawo, 3=góra
    gene_type=int, # Geny są typu całkowitego
    parent_selection_type="sss", # Selekcja rodziców: Steady State Selection
    keep_parents=5, # Liczba rodziców, którzy przechodzą do następnego pokolenia bez zmian
    crossover_type="single_point", # Typ krzyżowania: jednopunktowe
    mutation_type="random", # Typ mutacji: losowa -> klasyczna losowa zmiana kroku
    mutation_probability=0.15, # Prawdopodobieństwo mutacji
    random_seed=42, # Ustawienie ziarna losowości dla powtarzalności wyników
    save_solutions=True # Zapisywanie najlepszych rozwiązań z każdego pokolenia do analizy zbieżności
)

# 4. Uruchomienie uczenia 
print("Trwa ewolucja ścrieżki na jeziorze... Czekaj ...")
ga_instance.run()

# 5. Wyciągniecie najlepszego rozwiazania
solution, solution_fitness, solution_idx = ga_instance.best_solution()

#Dekodowanie liczb na ludzkie kierunki do wydruku w konsoli
kierunki = {0: "← lewo", 1: "↓ dół", 2: "→ prawo", 3: "↑ góra"}
sciezka_slownie = [kierunki[int(ruch)] for ruch in solution]

print("-" * 50)
print(f"Najlepsza znaleziona sekwencja ruchów:\n{sciezka_slownie}")
print(f"Wartość Fitness: {solution_fitness}")
print("-" * 50)

# 6. WYKRES ZBIEŻNOŚCI 
matplotlib.use('Qt5Agg') # Ustawienie backendu na Qt5Agg dla lepszej kompatybilności z Cyberpunkiem

plt.style.use("cyberpunk")
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 7)) # Trzy wykresy obok siebie

# ujednolicenie skali y dla obu wykresow (od 0 do 110 dla idealnego porównania)
max_y_limit = 110

#WYKRES 1: Zbieżność najlepszego osobnika
ax1.plot(ga_instance.best_solutions_fitness, color="#00ff41", linewidth=3, label="Najlepszy wynik")
ax1.set_title("1. Maksymalny Fitness w Pokoleniach", fontsize=12, color="cyan", pad=15)
ax1.set_xlabel("Pokolenie", color="cyan")
ax1.set_ylabel("Wartość Fitness", color="cyan")
ax1.set_ylim(0, max_y_limit) #ustawwienie stałej skali Y 
ax1.grid(True, linestyle='--', alpha=0.1) #delikatna siatka ułatwiajaca odczyt pokolenia 
ax1.legend(loc="upper left")
mplcyberpunk.make_lines_glow(ax1)
mplcyberpunk.add_underglow(ax1)

# WYKRES 2: Średni poziom całej populacji
#Wyciagamy średni fitness z każdego pokolenia
solutions_per_pop = ga_instance.sol_per_pop  #Liczba osobników w populacji -> generujemy dane per pokolenie
mean_fitness_history = []

#Łczymy oceny osobników w pakiety odpowiadajace kolejnym pokoleniom
for i in range(ga_instance.num_generations):
    start_idx = i * solutions_per_pop
    end_idx = start_idx + solutions_per_pop
    generation_fitness = ga_instance.solutions_fitness[start_idx:end_idx]
    mean_fitness_history.append(np.mean(generation_fitness))

# Rysujemy oryginalną poszarpaną linię średniej
ax2.plot(mean_fitness_history, color="#fe53bb", linewidth=1.5, alpha=0.5, label="Średnia populacji")
# Dodanie wygładzonej lini trendu (wielomian 3 stopnia), aby pokazac czysty kierunke ewolucji
pokolenia = np.arange(ga_instance.num_generations)
z = np.polyfit(pokolenia, mean_fitness_history, 3) # Dopasowanie wielomianu 3 stopnia
p = np.poly1d(z)
ax2.plot(pokolenia, p(pokolenia), color="#ff0055", linewidth=3, label="Główny trend uczenia")
ax2.set_title("2. Średni Fitness Całej Populacji", fontsize=12, color="cyan", pad=15)
ax2.set_xlabel("Pokolenie", color="cyan")
ax2.set_ylabel("Średni Fitness", color="cyan")
ax2.set_ylim(0, max_y_limit) #ustawwienie stałej skali Y
ax2.grid(True, linestyle='--', alpha=0.1) #delikatna siatka ułatwiajaca odczyt pokolenia
ax2.legend(loc="upper left")
mplcyberpunk.make_lines_glow(ax2)
mplcyberpunk.add_underglow(ax2)

# WYKRES 3: Mapa ciepła odwiedzanych komórek
# Logarytmujemy dane , aby lepiej uwypuklic rzadziej odwiedzane ściezki
log_heatmap = np.log1p(heatmap_data) # logarytm z (1 + liczba odwiedzin) dla lepszej wizualizacji
im = ax3.imshow(log_heatmap, cmap="magma", interpolation="nearest")

#Dodanie paska legendy kolorów pasującego do całości
cbar = fig.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)
cbar.set_label('Intensywność eksploracji (log)', color='cyan', fontsize=10)
cbar.ax.tick_params(labelsize=8, colors='cyan') 
ax3.set_title("3. Mapa Ciepła Eksploracji Środowiska", fontsize=12, color="cyan", pad=15)
ax3.set_xticks(range(4))
ax3.set_yticks(range(4))
ax3.set_xlabel("Kolumna siatki", color="cyan")
ax3.set_ylabel("Wiersz siatki", color="cyan")
ax3.grid(False) #wyłączamy siatkę, żeby nie zakłócała wizualizacji ciepła

plt.tight_layout()

# 7. URUCHOMIENIE WIZUALNE (Pokaz gry w okienku)
print("Uruchomienie pokazu graficznego w okienku...")
env_visual = gym.make('FrozenLake-v1',map_name="4x4", is_slippery=False, render_mode="human")
observation, info = env_visual.reset(seed=42)

# Krótka pauza na start, żeby zdążyć spojrzeć na okienko gry
time.sleep(1.5)

for action in solution:
    observation, reward, terminated, truncated, info = env_visual.step(int(action))
    
    #Zmieniamy predkosc
    time.sleep(0.6) #Pauza 0.6s między krokami, żeby było widać ruchy

    if terminated:
        break

#krótka pauza na koniec, żeby zdążyć zobaczyć efekt końcowy
time.sleep(2.0)

env_visual.close()
plt.show()


Trwa ewolucja ścrieżki na jeziorze... Czekaj ...
--------------------------------------------------
Najlepsza znaleziona sekwencja ruchów:
['→ prawo', '→ prawo', '↓ dół', '↓ dół', '↓ dół', '→ prawo', '↓ dół', '↓ dół', '→ prawo', '↑ góra', '↑ góra', '↓ dół']
Wartość Fitness: 106.0
--------------------------------------------------


TypeError: Artist.set_label() got an unexpected keyword argument 'color'